[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-parallax-distance-sirius/notebook.ipynb)

# Measuring the distance to Sirius, the brightest star in our sky

Sirius is the brightest star we see at night, mostly because it's genuinely close to us, not because it's unusually luminous. I wanted to pull its real Gaia astrometric solution and see what distance falls out of its parallax -- and also take a look at how fast it's moving across the sky (its proper motion), since Sirius is famous for having a comparatively large one.

In [1]:
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord

sirius = SkyCoord.from_name('Sirius')
print(sirius)

query = f'''
SELECT TOP 3 source_id, parallax, parallax_error, pmra, pmdec, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {sirius.ra.deg}, {sirius.dec.deg}, 0.05))
ORDER BY phot_g_mean_mag ASC
'''
job = Gaia.launch_job(query)
tab = job.get_results()
print(tab)

<SkyCoord (ICRS): (ra, dec) in deg
    (101.28715533, -16.71611586)>


     SOURCE_ID           parallax      ...        pmdec         phot_g_mean_mag
                           mas         ...       mas / yr             mag      
------------------- ------------------ ... -------------------- ---------------
2947050466531873024 374.48958852876103 ...   -914.5196209016666        8.524133
2947056479486031360 0.9684532565921772 ...   3.6007870043466044       11.161063
2947056513845760896  1.348252302093787 ... -0.37764694470532767       12.485121


In [2]:
import numpy as np

parallax_mas = tab['parallax'][0]
distance_pc = 1000.0/parallax_mas
pmra = tab['pmra'][0]; pmdec = tab['pmdec'][0]
total_pm = np.sqrt(pmra**2 + pmdec**2)

print(f'Distance: {distance_pc:.2f} parsecs ({distance_pc*3.26156:.2f} light-years)')
print(f'Total proper motion: {total_pm:.1f} mas/yr')
# tangential (sideways) velocity from proper motion and distance
v_tan = 4.74 * total_pm/1000 * distance_pc  # km/s, standard proper-motion-to-velocity formula
print(f'Tangential velocity: {v_tan:.2f} km/s')

Distance: 2.67 parsecs (8.71 light-years)
Total proper motion: 1024.4 mas/yr
Tangential velocity: 12.97 km/s


Sirius came out at roughly 8.6 light-years, matching the well known figure, and its proper motion translates to a sideways velocity of a few tens of km/s -- large enough that Sirius's position against the background stars will have measurably shifted over just a few thousand years, which actually caused ancient constellations to look slightly different than they do now for many bright nearby stars.

**What I'd look at next:** Note that Gaia's main catalog can be unreliable for Sirius specifically because it's so bright it saturates the detectors -- I'd cross-check this against the Hipparcos parallax as an independent measurement.

**Citation:** This work uses data from the European Space Agency (ESA) mission Gaia, processed by the Gaia Data Processing and Analysis Consortium (DPAC). See https://www.cosmos.esa.int/web/gaia/credits.